In [ ]:
# ============================================================
# 导入所需的库
# ============================================================
import os                          # 读取环境变量（比如 ChromaDB 的地址）
import re                          # 正则表达式，用于清理文本
import html                        # 处理 HTML 转义字符（比如 &amp; → &）

import torch                       # PyTorch，深度学习框架
import torch.nn.functional as F    # 提供 normalize 函数
import open_clip                   # CLIP 模型，用于图像/文字的视觉向量
import chromadb                    # 向量数据库客户端
from bs4 import BeautifulSoup      # 解析和清理 HTML 内容
from openai import OpenAI          # 调用 Ollama 的 OpenAI 兼容接口
from sentence_transformers import SentenceTransformer  # BERT 文字向量模型

from fastapi import FastAPI        # FastAPI 框架本身
from fastapi.middleware.cors import CORSMiddleware      # 允许跨域请求（前端访问后端需要）
from pydantic import BaseModel     # 定义请求数据的结构


# ============================================================
# 读取环境变量（这些值会在 docker-compose.yml 里设置）
# ============================================================
CHROMA_HOST = os.getenv("CHROMA_HOST", "localhost")   # ChromaDB 的主机名
CHROMA_PORT = int(os.getenv("CHROMA_PORT", "8000"))   # ChromaDB 的端口
OLLAMA_URL  = os.getenv("OLLAMA_URL", "http://localhost:11434/v1")  # Ollama 地址


# ============================================================
# 检测设备：有 GPU 就用 GPU，没有就用 CPU
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# 启动时加载模型（只加载一次，不然每次请求都加载会很慢）
# ============================================================

# BERT 模型：把文字转成向量（用于文字搜索）
bert_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# CLIP 模型：把文字/图片转成视觉向量（用于视觉搜索）
clip_model, _, _ = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="laion2b_s34b_b79k"
)
clip_model = clip_model.to(device).eval()
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")

# 连接 ChromaDB
chroma_client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
text_col  = chroma_client.get_collection("nasa_text_search")   # 存 BERT 向量的集合
image_col = chroma_client.get_collection("nasa_image_search")  # 存 CLIP 向量的集合

# 连接 Ollama（用于 RAG 的 LLM 总结）
llm_client = OpenAI(
    base_url=OLLAMA_URL,
    api_key="ollama",     # Ollama 不需要真实 key，随便填
    timeout=120.0
)


# ============================================================
# 创建 FastAPI 应用
# ============================================================
app = FastAPI()

# 允许前端（Streamlit）跨域访问后端
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


# ============================================================
# 定义请求体的数据结构（用 Pydantic）
# 相当于规定：前端发过来的 JSON 必须长这个样子
# ============================================================
class SearchRequest(BaseModel):
    query: str        # 用户输入的搜索词
    top_k: int = 5    # 返回几条结果，默认 5

class RAGRequest(BaseModel):
    query: str
    top_k: int = 3
    mode: str = "text"   # "text" 或 "image"


# ============================================================
# 工具函数：清理 HTML 文本
# 比如 "<p>Hello &amp; world</p>" → "Hello & world"
# ============================================================
def clean_text(raw: str) -> str:
    text = BeautifulSoup(raw or "", "html.parser").get_text(" ")
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:300]   # 只取前 300 字，避免太长


# ============================================================
# 接口 1：健康检查（用来确认服务是否正常运行）
# 访问 http://localhost:8001/ 会返回 {"status": "ok"}
# ============================================================
@app.get("/")
def root():
    return {"status": "ok"}


# ============================================================
# 接口 2：文字搜索（用 BERT 向量）
# POST http://localhost:8001/search/text
# 请求体：{"query": "pillars of creation", "top_k": 5}
# ============================================================
@app.post("/search/text")
def search_text(req: SearchRequest):
    # 第一步：把用户的文字转成 BERT 向量
    embedding = bert_model.encode(req.query).tolist()

    # 第二步：在 ChromaDB 里找最相似的向量
    results = text_col.query(
        query_embeddings=[embedding],
        n_results=req.top_k
    )

    # 第三步：返回结果
    return results


# ============================================================
# 接口 3：视觉搜索（用 CLIP 向量）
# POST http://localhost:8001/search/image
# ============================================================
@app.post("/search/image")
def search_image(req: SearchRequest):
    # 第一步：用 CLIP 把文字描述转成视觉向量
    tokens = clip_tokenizer([req.query]).to(device)
    with torch.no_grad():
        features = clip_model.encode_text(tokens)
    embedding = F.normalize(features, p=2, dim=-1).cpu().numpy().flatten().tolist()

    # 第二步：在 ChromaDB 里找最相似的视觉向量
    results = image_col.query(
        query_embeddings=[embedding],
        n_results=req.top_k
    )

    return results


# ============================================================
# 接口 4：RAG 搜索（检索 + LLM 总结）
# POST http://localhost:8001/search/rag
# ============================================================
@app.post("/search/rag")
def search_rag(req: RAGRequest):
    # 第一步：根据 mode 选择用哪种向量
    if req.mode == "image":
        tokens = clip_tokenizer([req.query]).to(device)
        with torch.no_grad():
            features = clip_model.encode_text(tokens)
        embedding = F.normalize(features, p=2, dim=-1).cpu().numpy().flatten().tolist()
        collection = image_col
        label = "visual characteristics"
    else:
        embedding = bert_model.encode(req.query).tolist()
        collection = text_col
        label = "scientific metadata"

    # 第二步：检索 top-3 结果（RAG 固定用 3 条）
    results = collection.query(
        query_embeddings=[embedding],
        n_results=3
    )

    ids   = results["ids"][0]
    metas = results["metadatas"][0]

    # 第三步：把检索结果整理成文字，作为 LLM 的上下文
    lines = []
    for meta in metas:
        title = clean_text(meta.get("title", ""))
        desc  = clean_text(meta.get("desc", meta.get("description", "")))
        lines.append(f"- {title}: {desc}")
    context_text = "\n".join(lines)

    # 第四步：调用 Llama3，让它根据上下文写一段总结
    try:
        response = llm_client.chat.completions.create(
            model="llama3",
            messages=[
                {
                    "role": "system",
                    "content": "You are a NASA scientist. Respond in English only. Maximum 2 sentences."
                },
                {
                    "role": "user",
                    "content": (
                        f'Summarize these NASA images for the query "{req.query}" '
                        f"based on their {label}:\n{context_text}"
                    )
                }
            ]
        )
        ai_overview = response.choices[0].message.content
    except Exception as e:
        ai_overview = f"AI summary unavailable: {str(e)}"

    # 第五步：返回 AI 总结 + 检索到的元数据
    return {
        "ai_overview": ai_overview,
        "hybrid_results": {
            "metadatas": metas,
            "ids": ids
        }
    }